# Does the frozen S1 + voice residual direction transfer to clause order?
**Primary experiment:** fixed-CoT readout at layer 18, chosen from the existing S1/voice shared-ablation figure. Reconstruct the base-subtracted directions from the original adapters and discovery splits. Freeze the S1+voice vector BEFORE measuring model 3. Do not select a new layer, direction, scale, or checkpoint using model-3 evaluation results.

Compare the third adapter with its unablated baseline and the unadapted base on 52 held-out pairs. Include S1-only, voice-only, model-3-own, and five seeded random rank-one directions. Apply shared ablation to the base too as a disruption control. For fixed readout, supply the space after `Final answer:` and verify greedy answer tokens before scoring. Keep all directions frozen at the original colon-position extraction site, including the own-rule control; recalibrate only centers on discovery examples at the supplied-space position. Free generation retains the original colon-position centers for comparison with the previous run. Restore the intervention center from each model's discovery activations; do not fit it on evaluation data.

The download contains actual residual tensors, split records, predictions, margins, controls, and an interactive 3D plot plus publication PNG/PDF. Local Azure critic scoring follows download. Recovery is a hypothesis, not an assumed result; single-label collapse or degradation is not restoration of normal reasoning.

Layer numbers are zero-based transformer block indices, matching the original notebook. Run interventions only at layer 18 and export fresh S1/voice layer-18 reference results. Existing layer-14 results are not rerun. No model-3 result is used to select the layer.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate matplotlib plotly pyyaml tqdm
from google.colab import files
from pathlib import Path
import json, hashlib, zipfile, os, sys, random, gc
ROOT = Path('/content/residual_third_rule'); ROOT.mkdir(exist_ok=True)
print('Upload residual_third_rule_bundle.zip (includes all three adapters; no API keys)')
uploaded = files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest = json.loads(Path('bundle_manifest.json').read_text())
for path, digest in manifest.items(): assert sha256(path) == digest, path
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
S1_ADAPTER = Path('checkpoints/s1')
VOICE_ADAPTER = Path('checkpoints/voice')
THIRD_ADAPTER = Path('checkpoints/clause')
PRIOR = Path('data/residual_s1_voice_transfer_05b')
prior = json.loads((PRIOR/'experiment.json').read_text())
assert sha256(S1_ADAPTER/'adapter_model.safetensors') == prior['s1_adapter_sha256']
assert sha256(VOICE_ADAPTER/'adapter_model.safetensors') == prior['voice_adapter_sha256']
from transformers import AutoTokenizer
for path in (S1_ADAPTER, VOICE_ADAPTER, THIRD_ADAPTER):
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(path)
LAYER = 18
DISCOVERY_N, SCAN_N, EVAL_N, SPLIT_SEED = 80, 80, 100, 0
OUT = Path('residual_third_rule_results_l18'); OUT.mkdir(exist_ok=True)
assert not (OUT/'experiment.json').exists(), 'Completed output already exists; download it or use a fresh runtime'


In [ ]:
import sys
from pathlib import Path

# make_minimal_negations.py does `from cot_utils import ...` (script-style).
sys.path.insert(0, str(Path("intervention").resolve()))
from evaluation.evaluate_ethics_morality import build_prompt
from cot_utils import split_sentences
from make_minimal_negations import swap_verdict

CANONICAL_S1 = {
    0: "The described action is perfectly fine.",
    1: "The described action is morally wrong.",
}


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def readout(scenario, cot):
    return f"{build_prompt({'scenario': scenario})} {cot}\nFinal answer:"


def voice_pairs(path):
    groups = {}
    for row in load_jsonl(path):
        groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
    pairs = []
    for pair_index, pair in sorted(groups.items()):
        if set(pair) != {"active", "passive"}:
            continue
        active, passive = pair["active"], pair["passive"]
        assert active["scenario"] == passive["scenario"]
        assert active["sentence_stances"] == passive["sentence_stances"]
        pairs.append({
            "pair_index": pair_index,
            "scenario": active["scenario"],
            "prompt": build_prompt({"scenario": active["scenario"]}),
            "pos_cot": active["chain_of_thought"],
            "neg_cot": passive["chain_of_thought"],
            "pos_text": readout(active["scenario"], active["chain_of_thought"]),
            "neg_text": readout(passive["scenario"], passive["chain_of_thought"]),
            "pos_label": 1,
            "neg_label": 0,
            "pos_cue": "active",
            "neg_cue": "passive",
            "gold": int(active.get("gold", active.get("final_answer", 0))),
            "flip": "voice",
        })
    return pairs


def make_s1_pair(row):
    sentences = list(row.get("sentences") or split_sentences(row["chain_of_thought"]))
    s1 = sentences[0]
    tail = " ".join(sentences[1:])
    stance = int(row.get("first_sentence_stance", row["sentence_stances"][0]))
    assert int(row["final_answer"]) == stance
    flipped = swap_verdict(s1, stance)
    flip_kind = "lexical"
    if flipped is None or flipped == s1:
        flipped = CANONICAL_S1[1 - stance]
        flip_kind = "canonical"
    flipped_cot = f"{flipped} {tail}".strip() if tail else flipped
    cot_by_stance = {stance: row["chain_of_thought"], 1 - stance: flipped_cot}
    return {
        "pair_index": int(row["index"]),
        "scenario": row["scenario"],
        "prompt": build_prompt({"scenario": row["scenario"]}),
        "pos_cot": cot_by_stance[1],
        "neg_cot": cot_by_stance[0],
        "pos_text": readout(row["scenario"], cot_by_stance[1]),
        "neg_text": readout(row["scenario"], cot_by_stance[0]),
        "pos_label": 1,
        "neg_label": 0,
        "pos_cue": "s1_wrong",
        "neg_cue": "s1_acceptable",
        "gold": int(row.get("gold", row["final_answer"])),
        "flip": flip_kind,
    }


def s1_pairs(path):
    return [make_s1_pair(row) for row in load_jsonl(path)]


def stratified_shuffle(pairs, seed):
    rng = random.Random(seed)
    by_key = {}
    for pair in pairs:
        by_key.setdefault(pair.get("flip", "none"), []).append(pair)
    mixed = []
    for key in sorted(by_key):
        bucket = by_key[key]
        rng.shuffle(bucket)
        mixed.extend(bucket)
    rng.shuffle(mixed)
    return mixed


def take_splits(pairs, *, discovery_n, scan_n, eval_n):
    assert len(pairs) >= discovery_n + scan_n + eval_n, (len(pairs), discovery_n, scan_n, eval_n)
    discovery = pairs[:discovery_n]
    scan = pairs[discovery_n:discovery_n + scan_n]
    eval_pairs = pairs[discovery_n + scan_n:discovery_n + scan_n + eval_n]
    return discovery, scan, eval_pairs


import random

voice_train_all = stratified_shuffle(
    voice_pairs(Path("data/training_data/synthetic_ethics_voice_paired_train.jsonl")),
    SPLIT_SEED,
)
voice_eval = voice_pairs(Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl"))
voice_discovery = voice_train_all[:DISCOVERY_N]
voice_scan = voice_train_all[DISCOVERY_N:DISCOVERY_N + SCAN_N]

s1_pool = stratified_shuffle(
    s1_pairs(Path("data/training_data/synthetic_ethics_cot_training_v2.jsonl"))
    + s1_pairs(Path("data/validation_data/synthetic_ethics_cot_val_v2.jsonl")),
    SPLIT_SEED,
)
s1_discovery, s1_scan, s1_eval = take_splits(
    s1_pool, discovery_n=DISCOVERY_N, scan_n=SCAN_N, eval_n=EVAL_N
)

assert len(voice_eval) == EVAL_N, len(voice_eval)
assert len(s1_eval) == EVAL_N, len(s1_eval)

TASKS = {
    "s1": {"discovery": s1_discovery, "scan": s1_scan, "eval": s1_eval, "adapter": S1_ADAPTER},
    "voice": {"discovery": voice_discovery, "scan": voice_scan, "eval": voice_eval, "adapter": VOICE_ADAPTER},
}
print("voice leftover train", len(voice_train_all) - DISCOVERY_N - SCAN_N)
print("S1 leftover pool   ", len(s1_pool) - DISCOVERY_N - SCAN_N - EVAL_N)
print("voice fit/scan/eval", len(voice_discovery), len(voice_scan), len(voice_eval))
print("S1    fit/scan/eval", len(s1_discovery), len(s1_scan), len(s1_eval))
print("S1 fit flips ", {k: sum(p["flip"] == k for p in s1_discovery) for k in ("lexical", "canonical")})
print("S1 scan flips", {k: sum(p["flip"] == k for p in s1_scan) for k in ("lexical", "canonical")})
print("S1 eval flips", {k: sum(p["flip"] == k for p in s1_eval) for k in ("lexical", "canonical")})
# Verify reconstructed original evaluation texts against the downloaded run.
for task in ('s1', 'voice'):
    old_rows = load_jsonl(PRIOR/f'{task}_val_unablated.jsonl')
    expected = {(r['index'], r['side']): r['chain_of_thought'] for r in old_rows}
    actual = {(p['pair_index'], side): p[f'{side}_cot'] for p in TASKS[task]['eval'] for side in ('pos', 'neg')}
    assert expected == actual, f'{task} split/text reconstruction differs from original run'

def clause_pairs(path):
    groups = {}
    for row in load_jsonl(path): groups.setdefault(row['pair_index'], {})[row['final_answer']] = row
    result = []
    for index, group in sorted(groups.items()):
        assert set(group) == {0, 1}
        a, b = group[1], group[0]
        assert a['scenario'] == b['scenario'] and a['sentence_stances'] == b['sentence_stances']
        result.append({'pair_index': index, 'scenario': a['scenario'], 'gold': a['gold'],
            'prompt': build_prompt(a), 'pos_cot': a['chain_of_thought'], 'neg_cot': b['chain_of_thought'],
            'pos_text': readout(a['scenario'], a['chain_of_thought']),
            'neg_text': readout(b['scenario'], b['chain_of_thought']), 'flip': 'clause_order'})
    return result
third_train = clause_pairs('data/training_data/synthetic_ethics_clause_order_paired_train.jsonl')
random.Random(0).shuffle(third_train)
third_eval = clause_pairs('data/validation_data/synthetic_ethics_clause_order_paired_eval.jsonl')
assert len(third_eval) == 52
assert not {p['scenario'].strip().casefold() for p in third_train} & {p['scenario'].strip().casefold() for p in third_eval}
TASKS['clause'] = {'discovery': third_train[:80], 'eval': third_eval, 'adapter': THIRD_ADAPTER}
(OUT/'splits.json').write_text(json.dumps({t: {s: TASKS[t][s] for s in ('discovery','eval')} for t in TASKS}, indent=2))


In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers

DEVICE = torch.device("cuda")
BATCH_SIZE = 8


def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids


def pair_texts(pairs, side):
    key = "pos_text" if side == "pos" else "neg_text"
    return [pair[key] for pair in pairs]


@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(rows) for rows in cached], dim=1)
    return activations, torch.cat(margins)


def collect_task(model, tokenizer, pairs):
    pos_h, pos_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "pos"))
    neg_h, neg_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "neg"))
    return {"pos_h": pos_h, "neg_h": neg_h, "pos_margin": pos_m, "neg_margin": neg_m}


def unload(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def collect_splits(model, tokenizer, task):
    return {
        split: collect_task(model, tokenizer, TASKS[task][split])
        for split in ("discovery", "scan", "eval")
    }



# Explicitly delete caller references when releasing models (not only local helper references).
base, adapter_acts = {}, {}
for task in ('s1', 'voice', 'clause'):
    tokenizer, model = load_model(BASE_MODEL, DEVICE)
    base[task] = {split: collect_task(model, tokenizer, TASKS[task][split]) for split in ('discovery','eval')}
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    tokenizer, model = load_model(str(TASKS[task]['adapter']), DEVICE)
    adapter_acts[task] = {split: collect_task(model, tokenizer, TASKS[task][split]) for split in ('discovery','eval')}
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    print('Collected', task, flush=True)
# Save raw activations so this experiment never depends on a surviving Colab runtime.
torch.save({'base': base, 'adapters': adapter_acts}, OUT/'paired_activations_all_layers.pt')


In [ ]:
def gap(bundle): return float(bundle['pos_margin'].mean() - bundle['neg_margin'].mean())
def pair_dod(task):
    a,b = adapter_acts[task]['discovery'], base[task]['discovery']
    return (a['pos_h'][:,LAYER]-a['neg_h'][:,LAYER]) - (b['pos_h'][:,LAYER]-b['neg_h'][:,LAYER])
def center(bundle): return torch.cat([bundle['pos_h'][:,LAYER],bundle['neg_h'][:,LAYER]]).mean(0)
d1 = F.normalize(pair_dod('s1').mean(0),dim=0)
d2 = F.normalize(pair_dod('voice').mean(0),dim=0)
_,singular,vh = torch.linalg.svd(torch.stack([d1,d2]),full_matrices=False)
shared = F.normalize(vh[0],dim=0)
if torch.dot(shared,d1)<0: shared = -shared
old_geometry = next(r for r in json.loads((PRIOR/'layer_geometry.json').read_text()) if r['layer']==LAYER)
cosine_error = abs(float(torch.dot(d1,d2))-old_geometry['cosine_s1_voice'])
assert cosine_error < 0.005, f'Prior geometry mismatch: {cosine_error}; investigate before interpreting transfer'
# Shared vector is frozen above. Model 3 contributes ONLY to its own diagnostic/control.
d3 = F.normalize(pair_dod('clause').mean(0),dim=0)
directions = {'s1': d1, 'voice': d2, 'shared': shared, 'own': d3}
for seed in range(5):
    directions[f'random_{seed}'] = F.normalize(torch.randn(d1.shape, generator=torch.Generator().manual_seed(seed)),dim=0)
centers = {t:center(adapter_acts[t]['discovery']) for t in TASKS}
base_center = center(base['clause']['discovery'])
torch.save({'layer':LAYER,'directions':directions,'centers':centers,'base_clause_center':base_center,
            'singular_values_s1_voice':singular},OUT/'frozen_directions.pt')
print('Prior cosine reconstruction error:',cosine_error)


In [ ]:
@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)



# Evaluate S1/voice shared projection at the selected layer only.
reference_results=[]
for task in ('s1','voice'):
    tokenizer,model=load_model(str(TASKS[task]['adapter']),DEVICE)
    margins={side:evaluate_projection(model,tokenizer,pair_texts(TASKS[task]['eval'],side),
        layer=LAYER,direction=shared,center=centers[task]) for side in ('pos','neg')}
    new_gap=float(margins['pos'].mean()-margins['neg'].mean())
    reference_results.append({'task':task,'layer':LAYER,'shared_gap':new_gap,
        'base_gap':gap(base[task]['eval']),'unablated_gap':gap(adapter_acts[task]['eval']),
        'shared_constructed_follow':float(torch.cat([margins['pos']>0,margins['neg']<0]).float().mean()),
        'pos_margins':margins['pos'].tolist(),'neg_margins':margins['neg'].tolist(),
        'readout':'legacy colon position; comparable to original S1/voice figure'})
    print(task,LAYER,'shared gap:',new_gap,flush=True)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()
(OUT/'s1_voice_reference_results.json').write_text(json.dumps(reference_results,indent=2))


In [ ]:
from copy import deepcopy
# Preserve historical extraction geometry. Change only the fixed-readout assay position.
third_eval=deepcopy(third_eval)
fit=deepcopy(TASKS['clause']['discovery'])
for pair in fit+third_eval:
    for side in ('pos','neg'):
        assert pair[side+'_text'].endswith('Final answer:')
        pair[side+'_text']+=' '
(OUT/'label_readout_splits.json').write_text(json.dumps({'discovery':fit,'eval':third_eval},indent=2))
diagnostics=[]
for name,path in [('base',BASE_MODEL),('unablated',str(THIRD_ADAPTER))]:
    tokenizer,model=load_model(path,DEVICE)
    assert tokenizer.encode(' 0',add_special_tokens=False)==tokenizer.encode(' ',add_special_tokens=False)+tokenizer.encode('0',add_special_tokens=False)
    for pair in third_eval[:2]:
        for side in ('pos','neg'):
            for space in (False,True):
                text=pair[side+'_text'] if space else pair[side+'_text'][:-1]
                inputs=tokenizer(text,return_tensors='pt').to(DEVICE)
                with torch.no_grad():
                    logits=model(**inputs).logits[0,-1].float()
                    output=model.generate(**inputs,max_new_tokens=8,do_sample=False,pad_token_id=tokenizer.eos_token_id,
                        temperature=None,top_p=None,top_k=None)
                ids=output[0,inputs['input_ids'].shape[1]:].tolist()
                z,o=label_token_ids(tokenizer)['0'],label_token_ids(tokenizer)['1']
                record={'model':name,'pair_index':pair['pair_index'],'side':side,'trailing_space':space,
                    'greedy_token_ids':ids,'greedy_text':tokenizer.decode(ids),
                    'restricted_digit_prediction':int(logits[o]>logits[z])}
                diagnostics.append(record);print(record)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()
(OUT/'token_position_diagnostic.json').write_text(json.dumps(diagnostics,indent=2))
# Confirm the model's first token after the supplied space really is a label token.
label_ids={z,o}
assert all(r['greedy_token_ids'][0] in label_ids for r in diagnostics if r['model']=='unablated' and r['trailing_space']), 'Unexpected token boundary; stop and investigate'

bundles={};label_centers={}
for name,path in [('base',BASE_MODEL),('unablated',str(THIRD_ADAPTER))]:
    tokenizer,model=load_model(path,DEVICE)
    discovered=collect_task(model,tokenizer,fit)
    label_centers[name]=torch.cat([discovered['pos_h'][:,LAYER],discovered['neg_h'][:,LAYER]]).mean(0)
    bundles[name]=collect_task(model,tokenizer,third_eval)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()
accuracy=float(torch.cat([bundles['unablated']['pos_margin']>0,bundles['unablated']['neg_margin']<0]).float().mean())
print('Corrected unablated paired rule accuracy:',accuracy)
assert accuracy>=0.95, 'Unablated rule not reproduced at label position; do not interpret ablation'

torch.save({'centers':label_centers,'eval':bundles,'layer':LAYER},OUT/'label_readout_bundles.pt')


In [ ]:
def write_arm(arm, bundle):
    records=[]
    for side,label in (('pos',1),('neg',0)):
        for pair,margin in zip(third_eval,bundle[side+'_margin']):
            # Two-label decision after the supplied space, validated against greedy tokens.
            prediction=int(float(margin)>0)
            records.append({'index':2*pair['pair_index']+label,'pair_index':pair['pair_index'],
                'arm':arm,'prompt':pair['prompt'],'scenario':pair['scenario'],'gold':pair['gold'],
                'chain_of_thought':pair[side+'_cot'],'final_answer':label,
                'clause_order':'cause_first' if label else 'cause_last','prediction':prediction,
                'logit_margin':float(margin),'prediction_protocol':'argmax over label tokens 0 and 1 after supplied space'})
    (OUT/f'{arm}.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in records))
    return records
write_arm('base',bundles['base'])
write_arm('unablated',bundles['unablated'])
for model_path,arms,local_center in [(BASE_MODEL,{'base_shared':shared},label_centers['base']),
                                     (str(THIRD_ADAPTER),directions,label_centers['unablated'])]:
    tokenizer,model=load_model(model_path,DEVICE)
    for arm,direction in arms.items():
        bundle={side+'_margin':evaluate_projection(model,tokenizer,pair_texts(third_eval,side),
            layer=LAYER,direction=direction,center=local_center) for side in ('pos','neg')}
        write_arm(arm,bundle)
        print('Finished',arm,flush=True)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()


In [ ]:
# Secondary test: project the last token at each generation step.
# This differs from the primary fixed-CoT readout test; report separately.
RUN_FREE_GENERATION = True
if RUN_FREE_GENERATION:
    import re
    from evaluation.evaluate_ethics_morality import parse_chain_of_thought
    source=load_jsonl('data/clause_order_baselines/base_ethics.jsonl')
    assert len(source)==100
    for arm,model_path,direction,local_center in [
        ('base',BASE_MODEL,None,base_center), ('base_shared',BASE_MODEL,shared,base_center),
        ('unablated',str(THIRD_ADAPTER),None,centers['clause']),
        ('shared',str(THIRD_ADAPTER),shared,centers['clause']),
        ('random_0',str(THIRD_ADAPTER),directions['random_0'],centers['clause'])]:
        tokenizer,model=load_model(model_path,DEVICE)
        handle=None
        if direction is not None:
            d=direction.to(DEVICE);c=local_center.to(DEVICE)
            def generation_hook(module,inputs,output):
                h=output[0] if isinstance(output,tuple) else output
                target=h[:,-1].float();patched=h.clone()
                patched[:,-1]=(target-((target-c)*d).sum(-1,keepdim=True)*d).to(h.dtype)
                return (patched,)+output[1:] if isinstance(output,tuple) else patched
            handle=transformer_layers(model)[LAYER].register_forward_hook(generation_hook)
        try:
            with (OUT/f'free_{arm}.jsonl').open('w') as f:
                for row in source:
                    inputs=tokenizer(row['prompt'],return_tensors='pt').to(DEVICE)
                    with torch.no_grad():
                        ids=model.generate(**inputs,do_sample=False,max_new_tokens=256,
                            pad_token_id=tokenizer.eos_token_id,temperature=None,top_p=None,top_k=None)
                    raw=tokenizer.decode(ids[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
                    match=re.search(r'\b(?:final answer|answer|label)\s*[:\-]\s*([01])\b',raw,re.I)
                    prediction=int(match[1]) if match else None
                    f.write(json.dumps({'index':row['index'],'prompt':row['prompt'],'gold':row['gold'],
                        'arm':arm,'chain_of_thought':parse_chain_of_thought(raw),'raw_generation':raw,
                        'prediction':prediction,'intervention_scope':'last token at every generation step'})+'\n')
        finally:
            if handle is not None:handle.remove()
        del model,tokenizer;gc.collect();torch.cuda.empty_cache()
        print('Free generations finished:',arm,flush=True)


## Geometry: four directions in their exact 3D span
An uncentered SVD supplies orthonormal display axes for the span of S1, voice, and clause-order vectors. The shared vector is a linear combination of S1 and voice. Thus this view preserves norms and angles up to numerical precision; axes are not individual neurons or anatomical coordinates. Including model 3 in the display basis does not change the frozen intervention vector. Geometry supports interpretation, not a causal claim on its own.


In [ ]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
names=['S1','Voice','Shared S1 + voice','Clause order']
colors=['#4385D3','#EA9343','#8B5FC4','#21A68D']
vectors=torch.stack([d1,d2,shared,d3])
_,_,basis=torch.linalg.svd(torch.stack([d1,d2,d3]),full_matrices=False)
xyz=(vectors@basis.T).numpy()
error=float((vectors-(vectors@basis.T)@basis).norm(dim=1).max())
assert error<1e-4,error
cosines=(vectors@vectors.T).numpy()
(OUT/'direction_geometry.json').write_text(json.dumps({'names':names,'cosines':cosines.tolist(),
    'coordinates':xyz.tolist(),'projection_error':error,'basis':'uncentered SVD span of S1, voice, clause; shared excluded from basis fit'},indent=2))
np.savez(OUT/'direction_geometry.npz',vectors=vectors.numpy(),basis=basis.numpy(),coordinates=xyz)
fig=go.Figure()
for name,color,(x,y,z) in zip(names,colors,xyz):
    fig.add_trace(go.Scatter3d(x=[0,x],y=[0,y],z=[0,z],mode='lines+markers+text',
        text=['',name],textposition='top center',name=name,line=dict(color=color,width=8),
        marker=dict(size=[2,5],color=color)))
    fig.add_trace(go.Cone(x=[x],y=[y],z=[z],u=[x],v=[y],w=[z],sizemode='absolute',sizeref=.10,
        anchor='tip',colorscale=[[0,color],[1,color]],showscale=False,showlegend=False))
fig.update_layout(title='Does a shared residual direction generalize to an unseen encoding?',
    template='plotly_white',height=750,margin=dict(l=20,r=20,b=20,t=70),
    scene=dict(aspectmode='cube',xaxis=dict(title='Span axis 1',range=[-1.15,1.15]),
        yaxis=dict(title='Span axis 2',range=[-1.15,1.15]),zaxis=dict(title='Span axis 3',range=[-1.15,1.15])))
fig.write_html(OUT/'residual_directions_3d.html',include_plotlyjs=True)
fig.show()
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11})
fig2=plt.figure(figsize=(12,5.5),layout='constrained')
ax=fig2.add_subplot(121,projection='3d')
for name,color,point in zip(names,colors,xyz):
    ax.quiver(0,0,0,*point,color=color,linewidth=2.5,arrow_length_ratio=.12)
    ax.text(*(point*1.08),name,color=color)
for setter in (ax.set_xlim,ax.set_ylim,ax.set_zlim):setter(-1.15,1.15)
ax.set(xlabel='Span axis 1',ylabel='Span axis 2',zlabel='Span axis 3',title=f'Residual directions at layer {LAYER}')
ax.set_box_aspect((1,1,1));ax.view_init(elev=23,azim=38)
heat=fig2.add_subplot(122);im=heat.imshow(cosines,vmin=-1,vmax=1,cmap='RdBu_r')
heat.set_xticks(range(4),names,rotation=35,ha='right');heat.set_yticks(range(4),names)
for i in range(4):
    for j in range(4):heat.text(j,i,f'{cosines[i,j]:.2f}',ha='center',va='center',color='white' if abs(cosines[i,j])>.65 else 'black')
heat.set_title('Cosines in the full 896-dimensional space')
fig2.colorbar(im,ax=heat,shrink=.7)
fig2.savefig(OUT/'residual_directions_3d.png',dpi=300)
fig2.savefig(OUT/'residual_directions_3d.pdf')
plt.show()


In [ ]:
import shutil
metadata={'base_model':BASE_MODEL,'layer':LAYER,'discovery_pairs_per_task':80,'evaluation_pairs':52,
    'shared_fit':'S1 and voice base-subtracted discovery means ONLY; no model-3 input',
    'third_direction':'model-3 training/discovery only; diagnostic and own-rule control',
    'projection':'h - dot(h-center,d)*d at final readout token; strength 1',
    'center':'fixed readout: supplied-space discovery mean; free generation: legacy colon-position discovery mean; never eval-fitted',
    'direction_extraction':'legacy colon position for all directions, including own control',
    'unablated_label_readout_accuracy':accuracy,
    'reference_results':reference_results,'random_seeds':list(range(5)),
    'prediction_protocol':'two-token 0/1 argmax after supplied space, checked against greedy labels; all arms use same rule',
    'layers_intervened':[LAYER],'bundle_manifest':manifest,
    'free_generation_enabled':RUN_FREE_GENERATION,
    'free_generation_scope':'last token at each decoding step; 256 new tokens; greedy',
    'interpretation':'Recovery toward base must be assessed alongside logit gaps, base agreement, accuracy and global label collapse. No success assumed.'}
(OUT/'experiment.json').write_text(json.dumps(metadata,indent=2))
archive=shutil.make_archive('/content/residual_third_rule_results_l18','zip',root_dir=OUT)
files.download(archive)
print('Extract into data/residual_third_rule_results_l18 locally, then run:')
print('python3 evaluation/score_residual_transfer.py --dir data/residual_third_rule_results_l18')
